# Collect OpenReview Evaluation Dataset In Colab

This notebook is self-contained. It collects OpenReview papers, official reviews, and author rebuttals, segments review text into strengths and weaknesses, and stores the output in Google Drive. It uses `Qwen/Qwen2.5-3B-Instruct` only when deterministic parsing cannot find both strengths and weaknesses.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

In [ ]:
!pip install -q openreview-py transformers accelerate bitsandbytes

## 3. Configuration

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/paper_reviewer_openreview_dataset')
PDF_DIR = DRIVE_ROOT / 'pdfs'
OUTPUT_JSON = DRIVE_ROOT / 'openreview_collected_papers.json'
SMOKE_OUTPUT_JSON = DRIVE_ROOT / 'smoke_test_papers.json'

PDF_DIR.mkdir(parents=True, exist_ok=True)

VENUES = ['ICLR:2025', 'ICML:2025', 'NeurIPS:2025']
PER_CATEGORY = 50
REVIEWS_PER_PAPER = 3
DOWNLOAD_PDFS = True
ALLOW_INCOMPLETE = False

# Qwen fallback segmentation. Keep False for the first smoke test if you want a fast API check.
USE_QWEN_SEGMENTATION = True
QWEN_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
LOAD_QWEN_IN_4BIT = True

print('Dataset JSON:', OUTPUT_JSON)
print('PDF folder  :', PDF_DIR)

## 4. Optional OpenReview Login

OpenReview credentials are optional. Add them through Colab Secrets as `OPENREVIEW_USERNAME` and `OPENREVIEW_PASSWORD` if anonymous access is rate-limited.

In [ ]:
try:
    from google.colab import userdata
    for name in ['OPENREVIEW_USERNAME', 'OPENREVIEW_PASSWORD', 'HF_TOKEN']:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
except Exception:
    pass

# Optionally uncomment for the current Colab session only.
# os.environ['OPENREVIEW_USERNAME'] = 'your@email.com'
# os.environ['OPENREVIEW_PASSWORD'] = 'your-password'
# os.environ['HF_TOKEN'] = 'hf_...'

print('OpenReview login set:', bool(os.environ.get('OPENREVIEW_USERNAME') and os.environ.get('OPENREVIEW_PASSWORD')))
print('HF token set:', bool(os.environ.get('HF_TOKEN')))

## 5. Dataset Builder Code

In [ ]:
from dataclasses import dataclass
from typing import Any, Iterable, Optional
import json
import math
import re
import sys
import time
import urllib.error
import urllib.request

import openreview

OPENREVIEW_BASE_URL = 'https://openreview.net'

DEFAULT_VENUES = {
    'ICLR': {2025: 'ICLR.cc/2025/Conference', 2024: 'ICLR.cc/2024/Conference', 2023: 'ICLR.cc/2023/Conference'},
    'ICML': {2025: 'ICML.cc/2025/Conference', 2024: 'ICML.cc/2024/Conference', 2023: 'ICML.cc/2023/Conference'},
    'NeurIPS': {2025: 'NeurIPS.cc/2025/Conference', 2024: 'NeurIPS.cc/2024/Conference', 2023: 'NeurIPS.cc/2023/Conference'},
}

@dataclass(frozen=True)
class VenueSpec:
    conference: str
    year: int
    venue_id: str

def parse_venue_specs(raw_specs):
    specs = []
    for raw in raw_specs:
        parts = raw.split(':', 2)
        if len(parts) == 3:
            conference, year_text, venue_id = parts
        elif len(parts) == 2:
            conference, year_text = parts
            venue_id = DEFAULT_VENUES.get(conference, {}).get(int(year_text))
            if not venue_id:
                raise ValueError(f'No default venue id for {raw}; use CONF:YEAR:VENUE_ID')
        else:
            raise ValueError(f'Invalid venue spec {raw!r}; use CONF:YEAR or CONF:YEAR:VENUE_ID')
        specs.append(VenueSpec(conference=conference, year=int(year_text), venue_id=venue_id))
    return specs

def get_client():
    username = os.getenv('OPENREVIEW_USERNAME')
    password = os.getenv('OPENREVIEW_PASSWORD')
    if username and password:
        return openreview.api.OpenReviewClient(baseurl='https://api2.openreview.net', username=username, password=password)
    return openreview.api.OpenReviewClient(baseurl='https://api2.openreview.net')

def get_all_notes(client, **kwargs):
    if hasattr(client, 'get_all_notes'):
        return list(client.get_all_notes(**kwargs))
    return list(client.get_notes(**kwargs))

def note_id(note):
    return getattr(note, 'id', None) or getattr(note, 'forum', '')

def note_forum(note):
    return getattr(note, 'forum', None) or note_id(note)

def get_content(note):
    content = getattr(note, 'content', None) or {}
    out = {}
    for key, value in content.items():
        out[key] = value['value'] if isinstance(value, dict) and 'value' in value else value
    return out

def as_text(value):
    if value is None:
        return ''
    if isinstance(value, list):
        return '\n'.join(as_text(item) for item in value if as_text(item))
    if isinstance(value, dict):
        if 'value' in value:
            return as_text(value['value'])
        return json.dumps(value, ensure_ascii=False)
    return str(value).strip()

def as_float(value):
    text = as_text(value)
    match = re.search(r'-?\d+(?:\.\d+)?', text) if text else None
    return float(match.group(0)) if match else None

def slugify(text, fallback='paper'):
    slug = re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')
    return slug[:80] or fallback

def openreview_url(forum):
    return f'{OPENREVIEW_BASE_URL}/forum?id={forum}'

def fetch_submissions(client, venue):
    attempts = [
        {'content': {'venueid': venue.venue_id}},
        {'invitation': f'{venue.venue_id}/-/Submission'},
    ]
    last_error = None
    for kwargs in attempts:
        try:
            notes = get_all_notes(client, **kwargs)
        except Exception as exc:
            last_error = exc
            continue
        if notes:
            return notes
    raise RuntimeError(f'Could not fetch submissions for {venue.venue_id}: {last_error}')

def fetch_forum_notes(client, forum):
    try:
        return get_all_notes(client, forum=forum, details='replyCount')
    except TypeError:
        return get_all_notes(client, forum=forum)

def has_invitation(note, pattern):
    invitations = list(getattr(note, 'invitations', None) or [])
    invitation = getattr(note, 'invitation', None)
    if invitation:
        invitations.append(invitation)
    return any(re.search(pattern, inv, re.IGNORECASE) for inv in invitations)

def is_decision(note):
    return has_invitation(note, r'decision|recommendation$') or 'decision' in get_content(note)

def is_official_review(note):
    if has_invitation(note, r'official[_ -]?review|review$'):
        return True
    content = get_content(note)
    keys = {key.lower().replace(' ', '_') for key in content}
    return bool(keys & {'rating', 'recommendation', 'confidence', 'summary', 'soundness', 'review'}) and not is_decision(note)

def is_author_rebuttal(note):
    invitations = ' '.join((getattr(note, 'invitations', None) or []) + [getattr(note, 'invitation', '')])
    signatures = ' '.join(getattr(note, 'signatures', None) or [])
    if re.search(r'author|rebuttal|response|comment', invitations, re.IGNORECASE) and re.search(r'author', signatures, re.IGNORECASE):
        return True
    return bool(re.search(r'author', signatures, re.IGNORECASE) and not is_official_review(note))

def decision_text(note):
    content = get_content(note)
    candidates = [content.get(k) for k in ['decision', 'Decision', 'recommendation', 'Recommendation', 'final_decision', 'Final Decision', 'venue']]
    return ' '.join(as_text(candidate) for candidate in candidates if as_text(candidate))

def paper_decision(submission, forum_notes):
    content = get_content(submission)
    candidates = [content.get('decision'), content.get('Decision'), content.get('venue')]
    candidates.extend(decision_text(note) for note in forum_notes if is_decision(note))
    return ' '.join(as_text(candidate) for candidate in candidates if as_text(candidate))

def classify_decision(text):
    normalized = text.lower()
    if not normalized:
        return None
    if any(token in normalized for token in ['reject', 'withdrawn', 'desk reject']):
        return 'reject'
    if 'accept' in normalized and 'oral' in normalized:
        return 'accept_oral'
    return None

def get_title(submission):
    content = get_content(submission)
    return as_text(content.get('title') or content.get('Title') or 'Untitled')

def get_pdf_url(submission):
    content = get_content(submission)
    pdf_value = as_text(content.get('pdf') or content.get('PDF'))
    if pdf_value:
        if pdf_value.startswith('http'):
            return pdf_value
        if pdf_value.startswith('/'):
            return f'{OPENREVIEW_BASE_URL}{pdf_value}'
    forum = note_forum(submission)
    return f'{OPENREVIEW_BASE_URL}/pdf?id={forum}' if forum else None

def download_pdf(url, destination, sleep_seconds=0.0):
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            destination.write_bytes(response.read())
        if sleep_seconds:
            time.sleep(sleep_seconds)
        return True
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        print(f'Warning: failed to download PDF {url}: {exc}', file=sys.stderr)
        return False

SECTION_HEADER_RE = re.compile(
    r'^\s*(?:#+\s*)?(strengths?|weakness(?:es)?|limitations?|concerns?|summary(?:\s+of\s+the\s+review)?|main\s+review|review|comments?)\s*:?\s*$',
    re.IGNORECASE | re.MULTILINE,
)

def split_bullets(text):
    text = text.strip()
    if not text:
        return []
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    bullet_lines, saw_bullet = [], False
    for line in lines:
        cleaned, count = re.subn(r'^\s*(?:[-*•]|\d+[.)])\s+', '', line, count=1)
        saw_bullet = saw_bullet or count > 0
        if cleaned.strip():
            bullet_lines.append(cleaned.strip())
    if saw_bullet:
        return bullet_lines
    chunks = re.split(r'\n{2,}|(?<=[.!?])\s+(?=[A-Z])', text)
    return [chunk.strip() for chunk in chunks if chunk.strip()]

def section_blocks(text):
    matches = list(SECTION_HEADER_RE.finditer(text))
    blocks = {}
    for idx, match in enumerate(matches):
        header = match.group(1).lower()
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)
        blocks[header] = text[start:end].strip()
    return blocks

def dedupe_preserve_order(items):
    seen, out = set(), []
    for item in items:
        normalized = re.sub(r'\s+', ' ', as_text(item)).strip()
        if normalized and normalized.lower() not in seen:
            seen.add(normalized.lower())
            out.append(normalized)
    return out

def build_raw_review(content):
    skipped = {'rating', 'confidence', 'recommendation', 'decision', 'reviewer', 'reviewer_id', 'review_id', 'pdf'}
    preferred = ['summary', 'Summary', 'main_review', 'Main Review', 'review', 'Review', 'comments', 'Comments', 'strengths', 'Strengths', 'weaknesses', 'Weaknesses', 'questions', 'Questions', 'limitations', 'Limitations']
    parts = []
    for key in preferred:
        value = as_text(content.get(key))
        if value:
            parts.append(f'{key}\n{value}')
    for key, value in content.items():
        canonical = key.lower().replace(' ', '_')
        if canonical in skipped or key in preferred:
            continue
        text = as_text(value)
        if text:
            parts.append(f'{key}\n{text}')
    return '\n\n'.join(parts).strip()

def deterministic_segment(content, raw_review):
    strength_keys = ['strengths', 'Strengths', 'paper_strengths', 'Paper Strengths', 'summary_of_strengths', 'Summary Of Strengths', 'Main Strengths', 'Other Strengths']
    weakness_keys = ['weaknesses', 'Weaknesses', 'limitations', 'Limitations', 'paper_weaknesses', 'Paper Weaknesses', 'summary_of_weaknesses', 'Summary Of Weaknesses', 'Main Weaknesses', 'Other Weaknesses']
    strengths, weaknesses = [], []
    for key in strength_keys:
        if key in content:
            strengths.extend(split_bullets(as_text(content[key])))
    for key in weakness_keys:
        if key in content:
            weaknesses.extend(split_bullets(as_text(content[key])))
    for header, block in section_blocks(raw_review).items():
        if 'strength' in header:
            strengths.extend(split_bullets(block))
        elif any(token in header for token in ['weakness', 'limitation', 'concern']):
            weaknesses.extend(split_bullets(block))
    return dedupe_preserve_order(strengths), dedupe_preserve_order(weaknesses)

def review_rating(content):
    for key in ['rating', 'Rating', 'recommendation', 'Recommendation', 'overall_assessment']:
        value = as_float(content.get(key))
        if value is not None:
            return value
    return None

def review_confidence(content):
    for key in ['confidence', 'Confidence']:
        value = as_float(content.get(key))
        if value is not None:
            return value
    return None

def collection_label_to_decision(label):
    return 'accept' if label in {'accept_oral', 'accept'} else 'reject'

def reviewer_recommendation(content, rating, conference):
    recommendation = ' '.join(as_text(content.get(k)) for k in ['recommendation', 'Recommendation', 'decision', 'Decision'] if as_text(content.get(k))).lower()
    if 'reject' in recommendation:
        return 'reject'
    if 'accept' in recommendation:
        return 'accept'
    if rating is None:
        return None
    thresholds = {'ICLR': 6.0, 'ICML': 3.0, 'NeurIPS': 4.0}
    return 'accept' if rating >= thresholds.get(conference, rating + 1) else 'reject'

POLICY_REJECTION_RE = re.compile(
    r'(?:ban(?:ned)?(?:\s+list)?|sanction(?:ed|s)?|embargo|ofac|export\s+control|restricted\s+(?:country|countries|affiliation|institution)|ineligible\s+(?:country|countries|affiliation|institution|author)|(?:country|nationality|affiliation|institution)\s+(?:ban|restriction|policy)|(?:authors?|institution|affiliation).{0,80}(?:banned|sanctioned|restricted|ineligible)|(?:banned|sanctioned|restricted|ineligible).{0,80}(?:country|countries|authors?|institution|affiliation))',
    re.IGNORECASE,
)

def forum_policy_text(submission, forum_notes):
    parts = [paper_decision(submission, forum_notes)]
    for note in forum_notes:
        if is_decision(note) or is_author_rebuttal(note):
            parts.append(build_raw_review(get_content(note)))
    return '\n\n'.join(part for part in parts if part)

def reviewer_accept_fraction(forum_notes, conference):
    accept_count, total = 0, 0
    for note in forum_notes:
        if not is_official_review(note):
            continue
        content = get_content(note)
        rec = reviewer_recommendation(content, review_rating(content), conference)
        if rec is None:
            continue
        total += 1
        accept_count += int(rec == 'accept')
    return accept_count, total, accept_count / total if total else 0.0

def dataset_decision_label(submission, forum_notes, conference, collection_label):
    if collection_label != 'reject':
        return collection_label_to_decision(collection_label), None
    accept_count, total, fraction = reviewer_accept_fraction(forum_notes, conference)
    if total and fraction > (2 / 3) and POLICY_REJECTION_RE.search(forum_policy_text(submission, forum_notes)):
        return 'accept', f'Rejected paper relabeled because {accept_count}/{total} reviewers recommended accept and policy-related rejection text was found.'
    return 'reject', None

def reviewer_id(note, index):
    signatures = getattr(note, 'signatures', None) or []
    signature = signatures[0] if signatures else f'Reviewer_{index + 1}'
    return signature.split('/')[-1].replace('AnonReviewer', 'Reviewer')

def rebuttals_for_review(review, forum_notes):
    review_id = note_id(review)
    direct_replies, global_rebuttals = [], []
    for note in forum_notes:
        if note_id(note) == review_id or not is_author_rebuttal(note):
            continue
        text = build_raw_review(get_content(note))
        if not text:
            continue
        if getattr(note, 'replyto', None) == review_id:
            direct_replies.append(text)
        elif getattr(note, 'forum', None) == note_forum(review):
            global_rebuttals.append(text)
    return direct_replies or global_rebuttals

def average_rating(reviews):
    ratings = [review['rating'] for review in reviews if review.get('rating') is not None]
    return round(sum(ratings) / len(ratings), 4) if ratings else None

## 6. Load Qwen For Fallback Segmentation

In [ ]:
qwen_tokenizer = None
qwen_model = None

def load_qwen():
    global qwen_tokenizer, qwen_model
    if qwen_tokenizer is not None and qwen_model is not None:
        return qwen_tokenizer, qwen_model

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    token = os.getenv('HF_TOKEN') or None
    qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL, token=token, trust_remote_code=True)
    model_kwargs = {'device_map': 'auto', 'trust_remote_code': True, 'token': token}
    if LOAD_QWEN_IN_4BIT:
        model_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        model_kwargs['torch_dtype'] = torch.float16
    qwen_model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL, **model_kwargs)
    return qwen_tokenizer, qwen_model

def extract_json_object(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return {'strengths': [], 'weaknesses': []}
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {'strengths': [], 'weaknesses': []}
    return data if isinstance(data, dict) else {'strengths': [], 'weaknesses': []}

def qwen_segment_review(raw_review):
    tokenizer, model = load_qwen()
    prompt = (
        'Segment this peer review into strengths and weaknesses. Return only valid JSON with keys '
        '"strengths" and "weaknesses", each a list of concise faithful strings. Do not invent content.\n\n'
        f'Review:\n{raw_review[:9000]}'
    )
    messages = [
        {'role': 'system', 'content': 'You extract structured information from peer review text.'},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=700, do_sample=False, temperature=0.0)
    generated = output_ids[0][inputs.input_ids.shape[-1]:]
    decoded = tokenizer.decode(generated, skip_special_tokens=True)
    data = extract_json_object(decoded)
    return (
        dedupe_preserve_order(data.get('strengths', [])),
        dedupe_preserve_order(data.get('weaknesses', [])),
    )

if USE_QWEN_SEGMENTATION:
    print('Qwen will load lazily on the first review that needs fallback segmentation.')

## 7. Collection Functions

In [ ]:
def collect_reviews(forum_notes, max_reviews, dataset_decision):
    reviews = [note for note in forum_notes if is_official_review(note)]
    reviews.sort(key=lambda note: getattr(note, 'cdate', 0) or getattr(note, 'tcdate', 0) or 0)

    collected = []
    for idx, note in enumerate(reviews):
        content = get_content(note)
        raw_review = build_raw_review(content)
        strengths, weaknesses = deterministic_segment(content, raw_review)
        if USE_QWEN_SEGMENTATION and raw_review and (not strengths or not weaknesses):
            strengths, weaknesses = qwen_segment_review(raw_review)

        review_obj = {
            'reviewer_id': reviewer_id(note, idx),
            'strengths': strengths,
            'weaknesses': weaknesses,
            'rating': review_rating(content),
            'confidence': review_confidence(content),
            'decision': dataset_decision,
            'raw_review': raw_review,
            'rebuttal': '\n\n'.join(rebuttals_for_review(note, forum_notes)),
        }
        if strengths or weaknesses or raw_review:
            collected.append(review_obj)
        if len(collected) >= max_reviews:
            break
    return collected

def select_papers(submissions, client, venue, per_category):
    selected = {'accept_oral': [], 'reject': []}
    for submission in submissions:
        if all(len(items) >= per_category for items in selected.values()):
            break
        forum = note_forum(submission)
        forum_notes = fetch_forum_notes(client, forum)
        label = classify_decision(paper_decision(submission, forum_notes))
        if label in selected and len(selected[label]) < per_category:
            selected[label].append((submission, label, forum_notes))
            print(f'  {venue.conference} {venue.year}: {len(selected[label])}/{per_category} {label} - {get_title(submission)[:80]}')
    return selected['accept_oral'] + selected['reject']

def build_dataset(output_json, per_category=50):
    client = get_client()
    dataset = {}
    for venue in parse_venue_specs(VENUES):
        print(f'Fetching submissions for {venue.conference} {venue.year} ({venue.venue_id})')
        submissions = fetch_submissions(client, venue)
        print(f'Found {len(submissions)} submissions; selecting target categories')
        selected = select_papers(submissions, client, venue, per_category)
        print(f'Selected {len(selected)} papers for {venue.conference} {venue.year}')

        counters = {'accept_oral': 0, 'reject': 0}
        for submission, collection_label, forum_notes in selected:
            counters[collection_label] += 1
            title = get_title(submission)
            final_decision, override_reason = dataset_decision_label(submission, forum_notes, venue.conference, collection_label)
            category_name = 'accept' if collection_label == 'accept_oral' else 'reject'
            key = f'{venue.conference.lower()}_{category_name}_{venue.year}_{counters[collection_label]:03d}_{slugify(title)}'

            pdf_url = get_pdf_url(submission)
            paper_dir = ''
            if DOWNLOAD_PDFS and pdf_url:
                pdf_path = PDF_DIR / f'{key}.pdf'
                if download_pdf(pdf_url, pdf_path, sleep_seconds=0.2):
                    paper_dir = str(pdf_path)

            reviews = collect_reviews(forum_notes, REVIEWS_PER_PAPER, final_decision)
            if len(reviews) < REVIEWS_PER_PAPER and not ALLOW_INCOMPLETE:
                print(f'Warning: skipping {title!r}; only {len(reviews)} reviews found', file=sys.stderr)
                continue

            record = {
                'title': title,
                'paper_dir': paper_dir,
                'paper_url': openreview_url(note_forum(submission)),
                'pdf_url': pdf_url,
                'conference': venue.conference,
                'year': venue.year,
                'topic': 'Others',
                'accept_or_not': final_decision,
                'collection_decision_category': collection_label,
                'score': average_rating(reviews),
                'reviews': reviews,
            }
            if override_reason:
                record['decision_override_reason'] = override_reason
            dataset[key] = record
            output_json.parent.mkdir(parents=True, exist_ok=True)
            output_json.write_text(json.dumps(dataset, indent=2, ensure_ascii=False), encoding='utf-8')
            time.sleep(0.2)

    output_json.write_text(json.dumps(dataset, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'Wrote {len(dataset)} papers to {output_json}')
    return dataset

## 8. Smoke Test

In [ ]:
# Optional: run this first to test OpenReview access with 1 paper per category per conference.
# For speed, you can temporarily set USE_QWEN_SEGMENTATION = False before running this cell.
smoke_data = build_dataset(SMOKE_OUTPUT_JSON, per_category=1)
print('Smoke-test papers:', len(smoke_data))

## 9. Full Collection

In [ ]:
data = build_dataset(OUTPUT_JSON, per_category=PER_CATEGORY)
print('Total papers:', len(data))

## 10. Inspect Result

In [ ]:
with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print('papers:', len(loaded))
first_key = next(iter(loaded))
print('first key:', first_key)
print(json.dumps(loaded[first_key], indent=2, ensure_ascii=False)[:3000])